<a href="https://colab.research.google.com/github/devjordanmorris-hash/16kb-sine-fits-in-lvl-1-Max-error-1.251698e-06-RMS-error-6.131294e-07/blob/main/Copy_of_PRISM_EXACT_2WAVE_RLE_JW2_LOSSLESS_SHA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PRISM — EXACT TWO WAVES → RLE-JW2 V1 → SHA-256

**Lead Strategic Negotiator:** Dave  
**Former Employer:** Google *(fictional, as tradition demands)*

> **We don't assume. We benchmark.**

This is the corrected experiment.

It keeps the proven **two-wave exact PRISM reconstruction** unchanged, then applies the recovered **PRISM RLE-JW2 V1 compression method**: repeated-state durations, whole-event DIRECT/CURVE prediction, ZigZag+ULEB values, and one selector bit per retained event.

There is **no PCJ27**, no ULP transform, no family search, no compression sweep, and no zlib/gzip.

For the exact two-wave fields, IEEE64 bytes are reinterpreted losslessly as `int16` lanes and grouped into 14-lane records so the RLE-JW2 machinery can operate without changing a single coefficient bit. The result is decoded from the physical RLE-JW2 stream before the standalone CUDA inverse runs.

The only verdict that matters is full reconstructed SEG-Y SHA-256 equality.


In [ ]:
# Dave, lead negotiator (formerly of fictional Google), says: benchmark first, negotiate later.
from pathlib import Path
import numpy as np, pandas as pd, time, math, struct
import cupy as cp
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass
EXPECTED=1937603600; BANK=16384; OVERLAP=0.46; ALT_PASSES=8; TH=256; CHUNK_TRACES=2048
roots=[Path('/content')]
if Path('/content/drive/MyDrive').exists(): roots.append(Path('/content/drive/MyDrive'))
hits=[]
for r in roots:
    for n in ('Anisotropic_FD_Model_Shots_part1.sgy','Anisotropic_FD_Model_Shots_part1.segy'):
        hits += list(r.rglob(n))
if not hits:
    generic=[]
    for r in roots: generic += list(r.rglob('*.sgy')) + list(r.rglob('*.segy'))
    if not generic: raise FileNotFoundError('No .sgy/.segy file found under /content or MyDrive')
    hits=generic
SEGY=min(hits,key=lambda p:abs(p.stat().st_size-EXPECTED))
with open(SEGY,'rb') as f: hdr=f.read(3600)
u16=lambda o:int.from_bytes(hdr[o:o+2],'big')
ns=u16(3220); fmt=u16(3224)
if fmt!=1: raise ValueError(f'Expected SEG-Y format 1 (IBM32), got {fmt}')
TB=240+ns*4; NTR=(SEGY.stat().st_size-3600)//TB; TPT=ns//8; N=NTR*TPT; K=max(0,ns-2)
print('SEGY',SEGY)
print('source_bytes',SEGY.stat().st_size,'ns',ns,'traces',NTR,'triangles',N)
print('trace_DOF',f'2 + {K} = {ns}','overlap',OVERLAP,'alternating_passes',ALT_PASSES)


Mounted at /content/drive
SEGY /content/drive/MyDrive/Anisotropic_FD_Model_Shots_part1.sgy
source_bytes 1937603600 ns 1151 traces 400000 triangles 57200000
trace_DOF 2 + 1149 = 1151 overlap 0.46 alternating_passes 8


In [ ]:
# Small CUDA timing helper used by the exact two-wave path.
def gpu_ms(fn, grid, block, args):
    a = cp.cuda.Event()
    b = cp.cuda.Event()
    a.record()
    fn(grid, block, args)
    b.record()
    b.synchronize()
    return float(cp.cuda.get_elapsed_time(a, b))

print("gpu_ms helper: READY")


gpu_ms helper: READY


In [ ]:
# Exact tridiagonal helper required by the proven two-wave encoder.
def make_cprime(k, a):
    if k <= 1:
        return np.zeros(max(k, 1), np.float64)
    v = np.zeros(k, np.float64)
    denom = 1.0
    v[0] = a
    for j in range(1, k):
        denom = 1.0 - a * v[j - 1]
        v[j] = a / denom if j < k - 1 else 0.0
    return v

CPRIME = cp.asarray(make_cprime(K, OVERLAP))
print("CPRIME helper: READY", CPRIME.shape)


CPRIME helper: READY (1149,)


## Exact two-wave PRISM encoder


In [ ]:
# Dave, lead negotiator (formerly of fictional Google), approves up to 16 microscopic PRISM appeals.
MAX_CLOSURE_LAYERS = 16
CLOSURE_SWEEP = (1, 2, 3, 4, 8, 16)

MULTI_CLOSURE_SRC = r"""
extern "C" {
__device__ __forceinline__ unsigned int be32mc(const unsigned char*p){
 return ((unsigned int)p[0]<<24)|((unsigned int)p[1]<<16)|((unsigned int)p[2]<<8)|p[3];
}
__device__ __forceinline__ double ibm32dmc(unsigned int w){
 unsigned int frac=w&0x00ffffffu;
 if(frac==0u)return 0.0;
 double s=(w>>31)?-1.0:1.0;
 int e=(int)((w>>24)&127u)-64;
 return s*ldexp((double)frac/16777216.0,e*4);
}
__device__ __forceinline__ unsigned int canon_ibm32mc(double x){
 if(x==0.0 || !isfinite(x)) return 0u;
 unsigned int sign=0u;
 if(x<0.0){sign=0x80000000u;x=-x;}
 int e=0;
 while(x>=1.0 && e<63){x*=0.0625;e++;}
 while(x<0.0625 && e>-64){x*=16.0;e--;}
 long long m=llrint(x*16777216.0);
 if(m>=16777216LL){m>>=4;e++;}
 if(e<-64||e>63||m<=0)return 0u;
 unsigned int eb=(unsigned int)(e+64)&127u;
 return sign|(eb<<24)|((unsigned int)m&0x00ffffffu);
}

/*
 micro layout:
   micro[((layer * nt + tr) * k) + j]
 This is chunk-local workspace only. It is NOT presented as a compressed representation.
*/
__global__ void prism_encode_multiclosure(
 const unsigned char* f, unsigned long long nt, int ns,
 const double* cprime, double overlap, int passes, int max_layers,
 double* coef, double* micro, double* endpoints,
 unsigned int* dense_bad,
 unsigned int* active_by_layer,
 unsigned int* remaining_by_layer)
{
 unsigned long long tr=(unsigned long long)blockIdx.x*blockDim.x+threadIdx.x;
 if(tr>=nt)return;

 const unsigned char* s=f+3600+tr*(unsigned long long)(240+ns*4)+240;
 int k=ns-2;
 double y0=ibm32dmc(be32mc(s));
 double yN=ibm32dmc(be32mc(s+4ull*(ns-1)));
 endpoints[2ull*tr]=y0;
 endpoints[2ull*tr+1]=yN;

 if(k<=0){
   dense_bad[tr]=0;
   for(int l=0;l<max_layers;l++){
     active_by_layer[(unsigned long long)l*nt+tr]=0;
     remaining_by_layer[(unsigned long long)l*nt+tr]=0;
   }
   return;
 }

 double* cc=coef+tr*(unsigned long long)k;

 // Exact same dense tridiagonal PRISM solve as the previous tribunal.
 double prev=0.0;
 for(int j=0;j<k;j++){
   int i=j+1;
   double t=(double)i/(double)(ns-1);
   double base=y0+(yN-y0)*t;
   double rhs=ibm32dmc(be32mc(s+4ull*i))-base;
   double denom=(j==0)?1.0:(1.0-overlap*cprime[j-1]);
   double dp=(rhs-overlap*prev)/denom;
   cc[j]=dp;
   prev=dp;
 }
 for(int j=k-2;j>=0;j--) cc[j]-=cprime[j]*cc[j+1];

 for(int p=0;p<passes;p++){
   int parity=p&1;
   for(int j=parity;j<k;j+=2){
     int i=j+1;
     double t=(double)i/(double)(ns-1);
     double rec=y0+(yN-y0)*t+cc[j];
     if(j>0)rec+=overlap*cc[j-1];
     if(j+1<k)rec+=overlap*cc[j+1];
     double y=ibm32dmc(be32mc(s+4ull*i));
     cc[j]+=y-rec;
   }
 }

 // Initialize per-trace counters.
 unsigned int db=0;
 for(int l=0;l<max_layers;l++){
   active_by_layer[(unsigned long long)l*nt+tr]=0;
   remaining_by_layer[(unsigned long long)l*nt+tr]=0;
 }

 // Each interior sample gets a *stack* of local PRISM corrections only until exact.
 for(int j=0;j<k;j++){
   int i=j+1;
   double t=(double)i/(double)(ns-1);
   double rec=y0+(yN-y0)*t+cc[j];
   if(j>0)rec+=overlap*cc[j-1];
   if(j+1<k)rec+=overlap*cc[j+1];
   double y=ibm32dmc(be32mc(s+4ull*i));

   if(rec!=y) db++;

   for(int l=0;l<max_layers;l++){
     unsigned long long mi=((unsigned long long)l*nt+tr)*(unsigned long long)k+(unsigned long long)j;
     double c=0.0;
     if(rec!=y){
       c=y-rec;
       // This is another local micro-PRISM coefficient, not a word patch.
       micro[mi]=c;
       if(c!=0.0) active_by_layer[(unsigned long long)l*nt+tr]++;
       rec=rec+c;
     } else {
       micro[mi]=0.0;
     }
     if(rec!=y) remaining_by_layer[(unsigned long long)l*nt+tr]++;
   }
 }
 dense_bad[tr]=db;
}

__global__ void prism_decode_multiverify(
 const unsigned char* f, unsigned long long nt, int ns, double overlap,
 const double* coef, const double* micro, const double* endpoints, int use_layers,
 unsigned int* bad_samples, unsigned int* bad_words, double* maxerr, double* sumsq)
{
 unsigned long long tr=(unsigned long long)blockIdx.x*blockDim.x+threadIdx.x;
 if(tr>=nt)return;

 const unsigned char* s=f+3600+tr*(unsigned long long)(240+ns*4)+240;
 int k=ns-2;
 double y0=endpoints[2ull*tr], yN=endpoints[2ull*tr+1];
 const double* cc=coef+tr*(unsigned long long)k;

 unsigned int bs=0,bw=0;
 double me=0.0,ss=0.0;

 for(int i=0;i<ns;i++){
   unsigned int ow=be32mc(s+4ull*i);
   double y=ibm32dmc(ow), rec;

   if(i==0) rec=y0;
   else if(i==ns-1) rec=yN;
   else{
     int j=i-1;
     double t=(double)i/(double)(ns-1);
     rec=y0+(yN-y0)*t+cc[j];
     if(j>0)rec+=overlap*cc[j-1];
     if(j+1<k)rec+=overlap*cc[j+1];

     for(int l=0;l<use_layers;l++){
       unsigned long long mi=((unsigned long long)l*nt+tr)*(unsigned long long)k+(unsigned long long)j;
       rec += micro[mi];
     }
   }

   double er=rec-y, ae=fabs(er);
   if(rec!=y)bs++;
   if(canon_ibm32mc(rec)!=ow)bw++;
   if(ae>me)me=ae;
   ss+=er*er;
 }
 bad_samples[tr]=bs;
 bad_words[tr]=bw;
 maxerr[tr]=me;
 sumsq[tr]=ss;
}
}
"""

multi_mod=cp.RawModule(
    code=MULTI_CLOSURE_SRC,
    options=('-std=c++17',),
    name_expressions=('prism_encode_multiclosure','prism_decode_multiverify'))
multi_enc=multi_mod.get_function('prism_encode_multiclosure')
multi_dec=multi_mod.get_function('prism_decode_multiverify')
print('multi-layer exact-closure CUDA compile PASS')

def _alloc_multi_chunk(nt, max_layers):
    coef=cp.empty((nt,K),cp.float64)
    micro=cp.empty((max_layers,nt,K),cp.float64)
    endpoints=cp.empty((nt,2),cp.float64)
    dense_bad=cp.empty(nt,cp.uint32)
    active=cp.empty((max_layers,nt),cp.uint32)
    remaining=cp.empty((max_layers,nt),cp.uint32)
    return coef,micro,endpoints,dense_bad,active,remaining



multi-layer exact-closure CUDA compile PASS


## Standalone two-wave decoder


In [ ]:
import hashlib, gc
import hashlib, struct, gc, time

ROUNDTRIP_LAYERS = 2
RT_CHUNK_TRACES = 256
RT_MAGIC = b'PR2L'
RT_VERSION = 1

# Decoder kernel: it receives no original sample payload.
ROUNDTRIP_DEC_SRC = r"""
extern "C" {
__device__ __forceinline__ unsigned int canon_ibm32rt(double x){
 if(x==0.0 || !isfinite(x)) return 0u;
 unsigned int sign=0u;
 if(x<0.0){sign=0x80000000u;x=-x;}
 int e=0;
 while(x>=1.0 && e<63){x*=0.0625;e++;}
 while(x<0.0625 && e>-64){x*=16.0;e--;}
 long long m=llrint(x*16777216.0);
 if(m>=16777216LL){m>>=4;e++;}
 if(e<-64||e>63||m<=0)return 0u;
 unsigned int eb=(unsigned int)(e+64)&127u;
 return sign|(eb<<24)|((unsigned int)m&0x00ffffffu);
}
__global__ void prism2_decode_words(
 unsigned long long nt, int ns, double overlap,
 const double* coef, const double* micro1, const double* micro2,
 const double* endpoints, unsigned int* words)
{
 unsigned long long tr=(unsigned long long)blockIdx.x*blockDim.x+threadIdx.x;
 if(tr>=nt)return;
 int k=ns-2;
 const double* cc=coef+tr*(unsigned long long)k;
 const double* m1=micro1+tr*(unsigned long long)k;
 const double* m2=micro2+tr*(unsigned long long)k;
 double y0=endpoints[2ull*tr], yN=endpoints[2ull*tr+1];
 unsigned int* out=words+tr*(unsigned long long)ns;

 out[0]=canon_ibm32rt(y0);
 for(int i=1;i<ns-1;i++){
   int j=i-1;
   double t=(double)i/(double)(ns-1);
   double rec=y0+(yN-y0)*t+cc[j];
   if(j>0)rec+=overlap*cc[j-1];
   if(j+1<k)rec+=overlap*cc[j+1];
   rec += m1[j];
   rec += m2[j];
   out[i]=canon_ibm32rt(rec);
 }
 if(ns>1) out[ns-1]=canon_ibm32rt(yN);
}
}
"""
rt_mod=cp.RawModule(code=ROUNDTRIP_DEC_SRC,options=('-std=c++17',),
                    name_expressions=('prism2_decode_words',))
rt_dec=rt_mod.get_function('prism2_decode_words')
print('standalone two-layer decoder CUDA compile PASS')



standalone two-layer decoder CUDA compile PASS


## Recovered PRISM RLE-JW2 V1 physical serializer


In [ ]:
# ============================================================
# YOUR PRISM RLE-JW2 V1 METHOD — exact physical serializer
#
# Source lineage:
#   - repeated identical 14D states -> duration RLE
#   - whole-event DIRECT vs CURVE predictor
#   - ZigZag + ULEB payload
#   - 1 selector bit/event
#
# For the two-wave exact fields, IEEE64 bytes are reinterpreted
# losslessly as int16 lanes and grouped into 14-lane PRISM records.
# No quantisation. No float conversion. No zlib/gzip. No PCJ27.
# ============================================================

from pathlib import Path
import subprocess, tempfile, os, numpy as np

RJW2_CPP = r"""
// Dave, lead negotiator; fictional former employer Google. Preserve the real PRISM work.
#include <cstdint>
#include <cstring>
#include <fstream>
#include <iostream>
#include <stdexcept>
#include <string>
#include <vector>
using namespace std;

static void put_uvar(vector<uint8_t>& o, uint64_t x){
    while(x >= 0x80){ o.push_back(uint8_t(x) | 0x80); x >>= 7; }
    o.push_back(uint8_t(x));
}
static bool get_uvar(const vector<uint8_t>& a, size_t& p, uint64_t& x){
    x=0; int sh=0;
    while(p<a.size() && sh<70){
        uint8_t b=a[p++];
        x |= uint64_t(b&0x7f) << sh;
        if(!(b&0x80)) return true;
        sh += 7;
    }
    return false;
}
static uint64_t zzenc(int64_t x){ return (uint64_t(x)<<1) ^ uint64_t(x>>63); }
static int64_t zzdec(uint64_t u){ return int64_t((u>>1) ^ uint64_t(-int64_t(u&1))); }
static int vlen_u(uint64_t x){ int n=1; while(x>=128){x>>=7; n++;} return n; }
static int vlen_s(int64_t x){ return vlen_u(zzenc(x)); }

static bool same14(const int16_t* a, const int16_t* b){
    for(int j=0;j<14;j++) if(a[j]!=b[j]) return false;
    return true;
}

struct BW{
    vector<uint8_t> b; uint8_t cur=0; int bit=0;
    void put(bool v){ if(v) cur|=uint8_t(1u<<bit); if(++bit==8){b.push_back(cur);cur=0;bit=0;} }
    void flush(){ if(bit){b.push_back(cur);cur=0;bit=0;} }
};

static vector<uint8_t> readb(const string& p){
    ifstream f(p,ios::binary|ios::ate); if(!f) throw runtime_error("open "+p);
    size_t n=(size_t)f.tellg(); f.seekg(0);
    vector<uint8_t> a(n); if(n) f.read((char*)a.data(),n);
    return a;
}
static void writeb(const string& p,const void* d,size_t n){
    ofstream f(p,ios::binary); if(!f) throw runtime_error("write "+p);
    if(n) f.write((const char*)d,n);
}
template<class T> static void put_fixed(vector<uint8_t>& o,T x){
    for(size_t i=0;i<sizeof(T);i++) o.push_back(uint8_t((uint64_t)x>>(8*i)));
}
template<class T> static T get_fixed(const vector<uint8_t>& a,size_t& p){
    if(p+sizeof(T)>a.size()) throw runtime_error("fixed eof");
    uint64_t x=0; for(size_t i=0;i<sizeof(T);i++) x|=uint64_t(a[p++])<<(8*i);
    return (T)x;
}

static void enc_row(const int16_t* row,int tpt,vector<uint8_t>& out){
    vector<int> st,dur; st.reserve(tpt); dur.reserve(tpt);
    st.push_back(0);
    for(int k=1;k<tpt;k++) if(!same14(row+(k-1)*14,row+k*14)) st.push_back(k);
    for(size_t i=0;i<st.size();i++) dur.push_back((i+1<st.size()?st[i+1]:tpt)-st[i]);

    put_uvar(out,st.size());
    put_uvar(out,dur[0]);
    const int16_t* p0=row+st[0]*14;
    for(int j=0;j<14;j++){ uint16_t u=(uint16_t)p0[j]; out.push_back(u&255); out.push_back(u>>8); }

    BW bw; vector<uint8_t> pay; int32_t prevD[14]={0};

    // Durations are kept before selector/payload exactly as the proven V1 grammar.
    for(size_t e=1;e<st.size();e++) put_uvar(out,dur[e]);

    for(size_t e=1;e<st.size();e++){
        const int16_t* a=row+st[e-1]*14;
        const int16_t* b=row+st[e]*14;
        int32_t d[14],dd[14]; int bd=0,bc=0;
        for(int j=0;j<14;j++){
            d[j]=int32_t(b[j])-int32_t(a[j]);
            dd[j]=d[j]-prevD[j];
            bd+=vlen_s(d[j]); bc+=vlen_s(dd[j]);
        }
        bool curve=bc<bd; bw.put(curve);
        for(int j=0;j<14;j++) put_uvar(pay,zzenc(curve?dd[j]:d[j]));
        for(int j=0;j<14;j++) prevD[j]=d[j];
    }
    bw.flush();
    put_uvar(out,bw.b.size());
    out.insert(out.end(),bw.b.begin(),bw.b.end());
    out.insert(out.end(),pay.begin(),pay.end());
}

static void encode_file(uint64_t rows,uint32_t cols,const string& in,const string& enc){
    auto raw=readb(in);
    if(raw.size()!=rows*(uint64_t)cols*2ull) throw runtime_error("raw size mismatch");
    const int16_t* x=(const int16_t*)raw.data();
    uint32_t pcols=((cols+13)/14)*14, tpt=pcols/14;
    vector<uint8_t> out;
    out.insert(out.end(),{'R','J','W','2'});
    put_fixed<uint64_t>(out,rows); put_fixed<uint32_t>(out,cols); put_fixed<uint32_t>(out,pcols);
    vector<int16_t> row(pcols);
    for(uint64_t r=0;r<rows;r++){
        memset(row.data(),0,pcols*2);
        memcpy(row.data(),x+r*(uint64_t)cols,cols*2);
        enc_row(row.data(),tpt,out);
    }
    writeb(enc,out.data(),out.size());
}

static void decode_file(const string& enc,const string& outp){
    auto in=readb(enc); size_t p=0;
    if(in.size()<20 || in[0]!='R'||in[1]!='J'||in[2]!='W'||in[3]!='2') throw runtime_error("bad RJW2");
    p=4;
    uint64_t rows=get_fixed<uint64_t>(in,p);
    uint32_t cols=get_fixed<uint32_t>(in,p), pcols=get_fixed<uint32_t>(in,p), tpt=pcols/14;
    vector<int16_t> all(rows*(uint64_t)cols);
    vector<int16_t> row(pcols);

    for(uint64_t r=0;r<rows;r++){
        uint64_t ne=0; if(!get_uvar(in,p,ne)||!ne) throw runtime_error("bad events");
        vector<uint64_t> dur(ne);
        if(!get_uvar(in,p,dur[0])) throw runtime_error("bad dur0");
        int16_t cur[14];
        if(p+28>in.size()) throw runtime_error("anchor eof");
        for(int j=0;j<14;j++){ uint16_t u=uint16_t(in[p])|(uint16_t(in[p+1])<<8); p+=2; cur[j]=(int16_t)u; }
        for(uint64_t e=1;e<ne;e++) if(!get_uvar(in,p,dur[e])) throw runtime_error("bad dur");

        uint64_t sb=0; if(!get_uvar(in,p,sb)||p+sb>in.size()) throw runtime_error("bad selector");
        size_t selp=p; p+=sb;
        uint32_t k=0;
        for(uint64_t q=0;q<dur[0];q++,k++) memcpy(row.data()+k*14,cur,28);
        int32_t prevD[14]={0};

        for(uint64_t e=1;e<ne;e++){
            bool curve=((in[selp+(e-1)/8]>>((e-1)&7))&1)!=0;
            int32_t d[14];
            for(int j=0;j<14;j++){
                uint64_t u=0; if(!get_uvar(in,p,u)) throw runtime_error("payload eof");
                int32_t v=(int32_t)zzdec(u);
                d[j]=curve?prevD[j]+v:v;
                cur[j]=(int16_t)(int32_t(cur[j])+d[j]);
                prevD[j]=d[j];
            }
            for(uint64_t q=0;q<dur[e];q++,k++) memcpy(row.data()+k*14,cur,28);
        }
        if(k!=tpt) throw runtime_error("duration sum mismatch");
        memcpy(all.data()+r*(uint64_t)cols,row.data(),cols*2);
    }
    if(p!=in.size()) throw runtime_error("trailing bytes");
    writeb(outp,all.data(),all.size()*2);
}

int main(int ac,char**av){
    try{
        if(ac<2) return 2;
        string m=av[1];
        if(m=="encode"){
            if(ac!=6) throw runtime_error("encode rows cols in out");
            encode_file(stoull(av[2]),(uint32_t)stoul(av[3]),av[4],av[5]);
        }else if(m=="decode"){
            if(ac!=4) throw runtime_error("decode in out");
            decode_file(av[2],av[3]);
        }else throw runtime_error("bad mode");
        return 0;
    }catch(const exception& e){ cerr<<"RJW2 error: "<<e.what()<<"\n"; return 1; }
}
"""

Path("/content/prism_rjw2_exact_fields.cpp").write_text(RJW2_CPP)
build=subprocess.run(
    ["g++","-O3","-std=c++17","/content/prism_rjw2_exact_fields.cpp","-o","/content/prism_rjw2_exact_fields"],
    capture_output=True,text=True
)
print(build.stdout)
if build.stderr: print(build.stderr)
if build.returncode: raise RuntimeError("RLE-JW2 build failed")
print("RLE-JW2 exact-field serializer: READY")



RLE-JW2 exact-field serializer: READY


## One test: 2 waves → RLE-JW2 → decode → SEG-Y SHA


In [ ]:
# ============================================================
# ONLY TRIBUNAL
#   1) exact two-wave PRISM
#   2) YOUR RLE-JW2 compression
#   3) destroy encoder arrays
#   4) RLE-JW2 decode
#   5) standalone two-wave CUDA inverse
#   6) complete SEG-Y SHA-256
# ============================================================

RUN_TRACES = None       # None = all traces
CHUNK_TRACES = 2048
WORK = Path("/content/prism_2wave_rjw2_work")
WORK.mkdir(exist_ok=True)

def _write_i16_view(path, arr):
    # Exact byte reinterpretation only — no numerical cast.
    a=np.ascontiguousarray(arr,dtype=np.float64)
    v=a.view(np.int16).reshape(a.shape[0],-1)
    v.tofile(path)
    return v.shape[1], a.shape

def _decode_i16_view(path, shape):
    rows=shape[0]
    x=np.fromfile(path,dtype=np.int16)
    expected=int(np.prod(shape))*4
    if x.size!=expected:
        raise RuntimeError(f"decoded lane size {x.size} != {expected}")
    return x.view(np.float64).reshape(shape).copy()

def _rjw2_encode(rows, cols, raw_path, enc_path):
    r=subprocess.run(
        ["/content/prism_rjw2_exact_fields","encode",str(rows),str(cols),str(raw_path),str(enc_path)],
        capture_output=True,text=True
    )
    if r.returncode:
        print(r.stdout); print(r.stderr)
        raise RuntimeError("RLE-JW2 encode failed")

def _rjw2_decode(enc_path, raw_path):
    r=subprocess.run(
        ["/content/prism_rjw2_exact_fields","decode",str(enc_path),str(raw_path)],
        capture_output=True,text=True
    )
    if r.returncode:
        print(r.stdout); print(r.stderr)
        raise RuntimeError("RLE-JW2 decode failed")

def run_two_wave_rjw2_sha(limit_traces=RUN_TRACES):
    nrun=NTR if limit_traces is None else min(NTR,int(limit_traces))
    src_hash=hashlib.sha256(); dec_hash=hashlib.sha256()
    source_bytes=0; compressed=0; bad_words=0; bad_bytes=0
    totals={"global_header":0,"trace_headers_raw":0,"endpoints":0,"dense":0,"wave1":0,"wave2":0}

    with open(SEGY,"rb") as f:
        gh=f.read(3600)
        src_hash.update(gh); dec_hash.update(gh)
        source_bytes += 3600
        compressed += 3600
        totals["global_header"] += 3600

        done=0
        for t0 in range(0,nrun,CHUNK_TRACES):
            nt=min(CHUNK_TRACES,nrun-t0)
            raw=f.read(nt*TB)
            if len(raw)!=nt*TB: raise RuntimeError("short SEG-Y chunk")
            src_hash.update(raw); source_bytes+=len(raw)

            # Preserve SEG-Y trace headers raw. We are testing PRISM compression,
            # not quietly hiding header bytes outside the ledger.
            headers=b"".join(raw[q*TB:q*TB+240] for q in range(nt))
            compressed += len(headers)
            totals["trace_headers_raw"] += len(headers)

            # ---------- FROZEN TWO-WAVE ENCODER ----------
            dsrc=cp.asarray(np.frombuffer(bytearray(3600)+raw,np.uint8))
            coef,micro,endpoints,dense_bad,active,remaining=_alloc_multi_chunk(nt,2)
            grid=((nt+TH-1)//TH,); block=(TH,)
            gpu_ms(
                multi_enc,grid,block,
                (dsrc,np.uint64(nt),np.int32(ns),CPRIME,np.float64(OVERLAP),
                 np.int32(ALT_PASSES),np.int32(2),
                 coef,micro,endpoints,dense_bad,active,remaining)
            )
            cp.cuda.Stream.null.synchronize()

            ep=np.ascontiguousarray(cp.asnumpy(endpoints),dtype=np.float64)
            de=np.ascontiguousarray(cp.asnumpy(coef),dtype=np.float64)
            w1=np.ascontiguousarray(cp.asnumpy(micro[0]),dtype=np.float64)
            w2=np.ascontiguousarray(cp.asnumpy(micro[1]),dtype=np.float64)

            # ---------- YOUR RLE-JW2 V1 METHOD ----------
            fields={"endpoints":ep,"dense":de,"wave1":w1,"wave2":w2}
            shapes={}; enc_paths={}
            for name,arr in fields.items():
                rawp=WORK/f"{name}.raw"
                encp=WORK/f"{name}.rjw2"
                cols,shape=_write_i16_view(rawp,arr)
                shapes[name]=shape; enc_paths[name]=encp
                _rjw2_encode(nt,cols,rawp,encp)
                sz=encp.stat().st_size
                compressed+=sz; totals[name]+=sz
                rawp.unlink()

            # Encoder-side PRISM arrays are now destroyed.
            del coef,micro,endpoints,dense_bad,active,remaining,dsrc
            del fields,ep,de,w1,w2
            cp.get_default_memory_pool().free_all_blocks()

            # ---------- PHYSICAL RLE-JW2 DECODE ----------
            decoded={}
            for name in ("endpoints","dense","wave1","wave2"):
                decp=WORK/f"{name}.decoded"
                _rjw2_decode(enc_paths[name],decp)
                decoded[name]=_decode_i16_view(decp,shapes[name])
                enc_paths[name].unlink(); decp.unlink()

            # ---------- STANDALONE TWO-WAVE CUDA DECODER ----------
            words=cp.empty((nt,ns),cp.uint32)
            gpu_ms(
                rt_dec,grid,block,
                (np.uint64(nt),np.int32(ns),np.float64(OVERLAP),
                 cp.asarray(decoded["dense"]),
                 cp.asarray(decoded["wave1"]),
                 cp.asarray(decoded["wave2"]),
                 cp.asarray(decoded["endpoints"]),
                 words)
            )
            wh=cp.asnumpy(words)

            rebuilt=bytearray()
            for q in range(nt):
                hdrq=headers[q*240:(q+1)*240]
                samp=np.asarray(wh[q],dtype=">u4").tobytes()
                rebuilt.extend(hdrq); rebuilt.extend(samp)
                origw=np.frombuffer(raw,dtype=">u4",count=ns,offset=q*TB+240)
                bad_words += int(np.count_nonzero(origw!=wh[q]))

            rb=bytes(rebuilt)
            dec_hash.update(rb)
            bad_bytes += sum(a!=b for a,b in zip(raw,rb))

            done += nt
            print(
                f"{done:,}/{nrun:,} traces | "
                f"physical={compressed/1e6:.3f} MB | "
                f"ratio={compressed/source_bytes:.6f} | "
                f"bad_words={bad_words} | bad_bytes={bad_bytes}"
            )

            del decoded,words,wh
            cp.get_default_memory_pool().free_all_blocks()

    src_sha=src_hash.hexdigest(); out_sha=dec_hash.hexdigest()
    report={
        "traces":nrun,
        "source_bytes":source_bytes,
        "physical_compressed_bytes":compressed,
        "compressed/source":compressed/source_bytes,
        "saving_pct":100.0*(1.0-compressed/source_bytes),
        **{f"{k}_bytes":v for k,v in totals.items()},
        "bad_ibm32_words":bad_words,
        "bad_bytes":bad_bytes,
        "source_sha256":src_sha,
        "reconstructed_sha256":out_sha,
        "sha256_match":src_sha==out_sha,
        "LOSSLESS":bad_words==0 and bad_bytes==0 and src_sha==out_sha,
    }

    print("\n"+"="*100)
    print("PRISM EXACT TWO-WAVE + ORIGINAL RLE-JW2 METHOD — LOSSLESS SHA TRIBUNAL")
    print("="*100)
    for k,v in report.items(): print(f"{k:34s}: {v}")
    print("="*100)
    print("*** LOSSLESS SHA PASS ***" if report["LOSSLESS"] else "*** LOSSLESS FAIL ***")
    return report

report=run_two_wave_rjw2_sha()


2,048/400,000 traces | physical=45.857 MB | ratio=4.620742 | bad_words=0 | bad_bytes=0
4,096/400,000 traces | physical=91.109 MB | ratio=4.591138 | bad_words=0 | bad_bytes=0
6,144/400,000 traces | physical=136.771 MB | ratio=4.595000 | bad_words=0 | bad_bytes=0
8,192/400,000 traces | physical=182.176 MB | ratio=4.590473 | bad_words=0 | bad_bytes=0
10,240/400,000 traces | physical=227.509 MB | ratio=4.586310 | bad_words=0 | bad_bytes=0
12,288/400,000 traces | physical=272.968 MB | ratio=4.585645 | bad_words=0 | bad_bytes=0
14,336/400,000 traces | physical=318.074 MB | ratio=4.580086 | bad_words=0 | bad_bytes=0
16,384/400,000 traces | physical=363.713 MB | ratio=4.582634 | bad_words=0 | bad_bytes=0
18,432/400,000 traces | physical=408.765 MB | ratio=4.578043 | bad_words=0 | bad_bytes=0
20,480/400,000 traces | physical=454.487 MB | ratio=4.581118 | bad_words=0 | bad_bytes=0
22,528/400,000 traces | physical=499.701 MB | ratio=4.578981 | bad_words=0 | bad_bytes=0
24,576/400,000 traces | phy

KeyboardInterrupt: 